# poblar en Colab

Calcula la población expuesta a la degradación de tierras a partir de un censo por radio, una cobertura del suelo y un mapa de degradación. Los criterios están en el archivo de método; los datos de cada ejecución se indican acá, archivo por archivo, con su ruta (el nombre de los archivos no importa).

1. Celda 3 (formulario): año, método y la ruta de cada insumo. Solo hacen falta los insumos de las etapas que se quieran correr.
2. Ejecutar las celdas en orden y revisar los controles (celda 5).
3. Mirar el informe y los mapas (celda 6).
4. Con dos años ejecutados, unir las tablas (celda 7).

Los rasters se procesan en el disco local de Colab y lo terminado se copia a Drive a medida que avanza. Si la sesión se corta, volver a ejecutar: con `ETAPAS = 'todas'` se corren las etapas que tienen sus insumos; para seguir desde donde quedó, indicar solo las que faltan.

In [ ]:
# 1. Instalación
# Se clona el repositorio y se instala desde el clon: el código y las configuraciones
# son de la misma versión, y la ejecución registra el commit en entorno.txt.
# Para reproducir una versión exacta, indicar una etiqueta o un commit en VERSION.
VERSION = 'main'
!rm -rf /content/poblar_repo
!git clone -q https://github.com/AugustoBellanich/poblar /content/poblar_repo
!git -C /content/poblar_repo checkout -q {VERSION}
!pip install -q -e /content/poblar_repo
# La instalación editable se activa recién al reiniciar el intérprete; con esto
# la librería se puede importar en la misma sesión, sin reiniciar.
import sys, importlib
if '/content/poblar_repo' not in sys.path:
    sys.path.insert(0, '/content/poblar_repo')
importlib.invalidate_caches()
# Si en esta sesión ya se había importado una versión anterior, se descarta para usar la nueva.
for _m in [m for m in sys.modules if m == 'poblar' or m.startswith('poblar.')]:
    del sys.modules[_m]
!git -C /content/poblar_repo log -1 --format='Versión: %h — %s (%cs)'

# Herramientas de GDAL (gdalwarp, gdal_rasterize) para las etapas 2 y 3
!which gdalwarp > /dev/null || (apt-get -qq update && apt-get -qq install -y gdal-bin > /dev/null)
!gdalinfo --version


In [ ]:
# 2. Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Parámetros de la ejecución
#@markdown **Período y método**
ANIO = 2022  #@param {type:"integer"}
METODO = '/content/poblar_repo/aplicaciones/prais2026/metodo.yaml'  #@param {type:"string"}
#@markdown Registro del período: huellas de los insumos oficiales y cifras publicadas. Desmarcar si se usan otros datos.
USAR_REGISTRO = True  #@param {type:"boolean"}
#@markdown ---
#@markdown **Insumos** — ruta de cada archivo, se llame como se llame. Las rutas relativas se buscan en `CARPETA_INSUMOS`. Vacío = valor por defecto del PRAIS para el año; `-` = no usar ese insumo.
CARPETA_INSUMOS = '/content/drive/MyDrive/PRAIS/poblar/insumos'  #@param {type:"string"}
TABLA_CENSO = ''  #@param {type:"string"}
RADIOS = ''  #@param {type:"string"}
REGIONES = ''  #@param {type:"string"}
COBERTURA = ''  #@param {type:"string"}
DEGRADACION = ''  #@param {type:"string"}
#@markdown ---
#@markdown **Salida**
SALIDA = '/content/drive/MyDrive/PRAIS/poblar/ejecuciones'  #@param {type:"string"}
#@markdown Nombre de la ejecución (vacío = `<aplicación>_<año>`). Con el mismo nombre se retoma la carpeta existente.
NOMBRE = ''  #@param {type:"string"}
#@markdown Etapas: `todas` o una lista separada por comas (`insumos, radios, grilla, dasimetria, cruce, publicacion, tablas`).
ETAPAS = 'todas'  #@param {type:"string"}

from pathlib import Path

POR_DEFECTO = {   # nombres de los insumos del PRAIS 2026 en CARPETA_INSUMOS
    'tabla_censo': f'tabla_censo{ANIO}.xlsx',
    'radios':      f'radios_{ANIO}.zip',
    'regiones':    'regiones_PRAIS.gpkg',
    'cobertura':   f'MB_prais_{2015 if ANIO == 2010 else 2023}_100m.tif',
    'degradacion': f'degradacion_{"referencia" if ANIO == 2010 else "reporte"}_{ANIO}.tif',
}
indicados = {'tabla_censo': TABLA_CENSO, 'radios': RADIOS, 'regiones': REGIONES,
             'cobertura': COBERTURA, 'degradacion': DEGRADACION}
INSUMOS = {}
for rol, valor in indicados.items():
    valor = valor.strip() or POR_DEFECTO[rol]
    if valor != '-':
        INSUMOS[rol] = str(Path(CARPETA_INSUMOS) / valor)   # si valor es absoluta, queda igual

METODO = Path(METODO)
REGISTRO = METODO.parent / 'periodos' / f'{ANIO}.yaml' if USAR_REGISTRO else None
if REGISTRO is not None and not REGISTRO.exists():
    print(f'No hay registro para {ANIO}: se ejecuta sin él.')
    REGISTRO = None
SALIDA = Path(SALIDA)
TRABAJO = Path('/content/trabajo')   # disco local de Colab
ETAPAS = 'todas' if ETAPAS.strip() == 'todas' else [e.strip() for e in ETAPAS.split(',') if e.strip()]

# Verificación previa: que existan los archivos indicados
faltan = [f'{rol}: {ruta}' for rol, ruta in INSUMOS.items() if not Path(ruta).exists()]
if faltan:
    raise FileNotFoundError('No se encontraron estos insumos:\n  ' + '\n  '.join(faltan))
for rol, ruta in INSUMOS.items():
    print(f'{rol:12} {ruta}')

In [ ]:
# 4. Ejecución
# Si un control bloqueante falla, la ejecución se detiene y se informa el motivo;
# los controles y el balance quedan igual en la carpeta para revisarlos.
from poblar import ejecutar
from poblar.trazabilidad import ControlFallido

try:
    ejecucion = ejecutar(METODO, anio=ANIO, insumos=INSUMOS, registro=REGISTRO,
                         salida=SALIDA, etapas=ETAPAS, trabajo=TRABAJO,
                         nombre=NOMBRE or None)
    CARPETA = ejecucion.carpeta
    print('\nEjecución completa:', CARPETA)
except ControlFallido as e:
    CARPETA = e.carpeta
    print('\nLa ejecución se detuvo:', e)
    print('Carpeta:', CARPETA)

In [ ]:
# 5. Controles y balance
import pandas as pd
pd.set_option('display.max_colwidth', 120)
controles = pd.read_csv(CARPETA / 'controles.csv')
print(f"Controles: {len(controles)} | sin cumplir: {(~controles['ok']).sum()}")
display(controles[['etapa', 'nombre', 'ok', 'detalle', 'bloqueante']])
display(pd.read_csv(CARPETA / 'balance.csv'))

## Resultados

Cada ejecución deja un informe (`informe_<nombre>.html`, por ejemplo `informe_prais2026_2022.html`) con los controles, las cifras principales, mapas de vista rápida y las tablas de indicadores. La celda siguiente lo muestra acá; también se puede abrir desde Drive.

Para mirar un producto en particular: `vista.mapa(CARPETA, producto, region=...)`. `vista.productos(CARPETA)` lista los disponibles.

In [ ]:
# 6. Informe y vistas rápidas
from IPython.display import HTML
from poblar import informe, vista

#@markdown Regenerar el informe con la versión instalada de poblar (por ejemplo, después de actualizarla). Tarda uno o dos minutos por los mapas.
REGENERAR = False  #@param {type:"boolean"}
ruta_informe = informe.ruta(CARPETA)          # informe_<nombre>.html
if REGENERAR or not ruta_informe.exists():   # también para ejecuciones anteriores al informe
    ruta_informe = informe.generar(CARPETA)
display(HTML(ruta_informe.read_text(encoding='utf-8')))

#@markdown Vista de un producto (`poblacion`, `expuesta`, `degradacion`, `cobertura`, `nbi`, ...). Región vacía = todo el país.
PRODUCTO = 'expuesta'  #@param {type:"string"}
REGION = ''  #@param {type:"string"}
print('Productos disponibles:', ', '.join(vista.productos(CARPETA)))
vista.mapa(CARPETA, PRODUCTO, region=REGION or None);

## Tablas de dos períodos

Con la etapa `tablas` ejecutada en los dos años, une las tablas con una columna por período.


In [ ]:
# 7. Tablas de dos períodos
from poblar.tablas import unir_periodos

ANIOS = (2010, 2022)
BASE = CARPETA.name.rsplit('_', 1)[0]                 # por ejemplo 'prais2026'
EJEC = {a: SALIDA / f'{BASE}_{a}' for a in ANIOS}
DEST = SALIDA / f'{BASE}_tablas'
for nombre, t in unir_periodos(EJEC, DEST).items():
    print(nombre); display(t)
print('Tablas en', DEST)
